# 🧗 Lab 7: Solving MountainCarContinuous with A3C in Gymnasium
In this tutorial, you will:
- Understand the MountainCarContinuous-v0 environment
- Implement an Asynchronous Advantage Actor-Critic (A3C) agent
- Train and visualize the agent’s behavior
- Reflect on how A3C differs from DQN, Double DQN, and DRQN

In [ ]:
# ✅ Step 1: Install and import required packages
!pip install gymnasium torch matplotlib -q
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt
import multiprocessing
import time

## 🧠 Step 2: Define the Actor-Critic Neural Network

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.common = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.actor = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Tanh()
        )
        self.critic = nn.Linear(128, 1)
    def forward(self, x):
        x = self.common(x)
        return self.actor(x), self.critic(x)

## 🧵 Step 3: Define the A3C Worker Function

In [ ]:
def worker(global_model, optimizer, env_name, global_rewards, id):
    env = gym.make(env_name)
    local_model = ActorCritic(env.observation_space.shape[0], env.action_space.shape[0])
    for episode in range(50):
        state, _ = env.reset()
        done = False
        ep_reward = 0
        while not done:
            state_tensor = torch.FloatTensor(state)
            mu, value = local_model(state_tensor)
            dist = Normal(mu, torch.tensor(0.1))
            action = dist.sample().numpy()
            next_state, reward, done, _, _ = env.step(action)
            ep_reward += reward
            state = next_state
        global_rewards.append(ep_reward)
        print(f"Worker {id} | Episode {episode} | Reward: {ep_reward:.2f}")

## ▶️ Step 4: Run Multiple A3C Workers and Plot

In [ ]:
if __name__ == '__main__':
    env_name = 'MountainCarContinuous-v0'
    global_model = ActorCritic(2, 1)
    global_model.share_memory()
    optimizer = optim.Adam(global_model.parameters(), lr=1e-3)
    manager = multiprocessing.Manager()
    global_rewards = manager.list()
    processes = []
    for id in range(2):
        p = multiprocessing.Process(target=worker, args=(global_model, optimizer, env_name, global_rewards, id))
        p.start()
        processes.append(p)
    for p in processes:
        p.join()
    plt.plot(global_rewards)
    plt.title('A3C - MountainCarContinuous')
    plt.xlabel('Episodes')
    plt.ylabel('Reward')
    plt.show()